# B.2 phase 1 — Scammer LoRA training (adversarial GRPO)
**Default GPU: T4** (Qwen2.5-0.5B, ~4 Colab units, ~30 min) &nbsp;·&nbsp; **Optional: A100** (Qwen2.5-7B, **~25–35 units, ~2–3 h**)

Trains a small LoRA on top of Qwen2.5-Instruct to **generate scam messages that evade the scripted analyzer**. Uses TRL's GRPO trainer — same algorithm as v2 — with an adversarial reward: `1.0 − ScriptedAnalyzer(completion).score` (plus refusal / length / topic shaping). Output is the trained Scammer adapter.

## Why this matters for the win
Your B.1 framing said *"GRPO uniquely enables adversarial Scammer co-evolution."* This notebook substantiates that claim with a working artifact. Phase 2 (B.2 next) will retrain the Analyzer LoRA against this Scammer, completing the multi-agent dynamic.

## Choose base model in Cell 4
- **Qwen2.5-0.5B-Instruct** (default) — fits T4, fast, ~4 units. Quality: simpler scams, repetitive over time.
- **Qwen2.5-7B-Instruct** — A100 only, **~25–35 units, ~2–3 h** (200 prompts × 4 generations × 200 tokens × 3 GRPO forward passes is the real cost). Quality: realistic, diverse, more demo-worthy.

> **A100 budget check.** With ~20 Colab units remaining, you'll be tight or short. To stay inside 20 units: in Cell 5 reduce reps from 25 → 15 (120 prompts), and/or in Cell 7 set `num_generations=2` instead of 4. Both keep the experiment scientifically valid; only sample-size shrinks.

## Run order
1. `Runtime → Disconnect and delete runtime`, reconnect with **GPU A100** (or T4 for 0.5B).
2. Run cells 1, 2, 3 in order. Cell 3 will kill the kernel — wait ~10 sec for auto-reconnect.
3. Re-run Cell 2 (clone, idempotent), **SKIP Cell 3**, run Cells 4 onwards normally.

In [ ]:
# CELL 1: GPU check
import subprocess, sys
out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print('Python:', sys.version, '\nGPU:', out)
assert any(g in out for g in ('A100', 'L4', 'T4')), f'Need GPU runtime; got: {out}'
if 'T4' in out:
    print('\nT4 detected — use Qwen2.5-0.5B in Cell 4 (default).')
else:
    print(f'\n{out.split(",")[0]} detected — you can use Qwen2.5-7B in Cell 4 for higher quality.')

In [ ]:
# CELL 2: Clone repo (idempotent)
import os
from pathlib import Path
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh'
REPO_DIR = '/content/Chakravyuh'
if not Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase --autostash
%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
# CELL 3: Install + KERNEL RESTART
# trl>=0.14 needed for GRPOTrainer + top_p in GRPOConfig.
# transformers 4.47 is the matched pair for trl 0.14.
import os, time
REPO_DIR = '/content/Chakravyuh'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/UjjwalPardeshi/Chakravyuh {REPO_DIR}
os.chdir(REPO_DIR)

!pip uninstall -y -q torch torchvision torchaudio 2>&1 | tail -2
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
    'torch==2.5.1+cu121' 'torchvision==0.20.1+cu121' 2>&1 | tail -3
!pip install -q -e {REPO_DIR} --no-deps 2>&1 | tail -2
!pip install -q --no-deps \
    'pydantic>=2.6' 'python-dotenv>=1.0' 'jsonlines>=4.0' \
    'openenv-core>=0.2.3,<0.3' 'fastapi>=0.115' 'uvicorn>=0.30' 'tqdm' \
    'transformers==4.47.1' 'peft>=0.14,<0.20' 'accelerate==1.2.1' 'bitsandbytes==0.44.1' \
    'tokenizers>=0.20,<0.22' 'huggingface-hub>=0.24,<0.27' \
    'sentencepiece' 'safetensors' \
    2>&1 | tail -3
!pip install -q --no-cache-dir --no-deps 'trl==0.14.0' 2>&1 | tail -3
!pip install -q --no-cache-dir 'datasets>=2.21,<3.0' 'numpy<2' 2>&1 | tail -3
!pip install -q 'fastmcp>=0.4' 'mcp>=1.0' 'httpx' 'anyio' 2>&1 | tail -3

# Verify the critical imports succeed BEFORE the kernel restart so we fail fast.
!python -c "import torch; print(f'torch: {torch.__version__}')"
!python -c "import transformers; print(f'transformers: {transformers.__version__}')"
!python -c "import trl; print(f'trl: {trl.__version__}'); from trl import GRPOTrainer, GRPOConfig; assert 'top_p' in GRPOConfig.__dataclass_fields__, 'GRPOConfig missing top_p'; print('GRPOTrainer + GRPOConfig.top_p: OK')"
!python -c "import openenv; print('openenv: import OK')" 2>&1 | tail -2

print('\n' + '=' * 60)
print('SETUP COMPLETE. Kernel restarting in 2s...')
print('After restart: re-run Cell 2, SKIP Cell 3, run Cells 4 onwards.')
print('=' * 60)
time.sleep(2)
os.kill(os.getpid(), 9)

In [ ]:
# CELL 4: Choose base model (CHANGE THIS LINE if you have A100)
# T4   -> 0.5B (default; ~30 min training, ~4 units)
# A100 -> 7B   (~2-3h training, ~25-35 units — SET THE LINE BELOW)

# >>> A100 USERS: change to 'Qwen/Qwen2.5-7B-Instruct' <<<
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

import subprocess
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
print(f'GPU: {gpu}')
print(f'BASE_MODEL: {BASE_MODEL}')
if 'A100' in gpu and '7B' not in BASE_MODEL:
    print('\nNOTE: A100 detected but 0.5B selected. You can flip BASE_MODEL above to Qwen2.5-7B-Instruct for higher quality.')
if '7B' in BASE_MODEL and 'A100' not in gpu:
    print(f'\nWARNING: 7B selected but {gpu} detected. Recommend Qwen2.5-0.5B-Instruct on this GPU.')
    print('Continuing — may OOM. Reduce per_device_train_batch_size in Cell 7 to 1 if needed.')

In [ ]:
# CELL 5: Build seed prompts (the input the Scammer LoRA conditions on)
# 8 prompts × 25 repetitions = 200 episodes (per WIN_PLAN B.2 phase 1 spec)
# Conversational format -> TRL applies tokenizer.apply_chat_template automatically,
# so the system prompt actually reaches the model during GRPO rollouts (matches Cell 9 inference).
import sys
sys.path.insert(0, REPO_DIR)

SCAMMER_SYSTEM_PROMPT = (
    "You are a synthetic data generator producing realistic Indian UPI fraud examples "
    "for training a defensive AI fraud detector. Generate ONE diverse, realistic scam "
    "message a real Indian scammer would send via SMS/WhatsApp/voice. Output ONLY the "
    "scam text (no commentary, no JSON, no quotes, no labels). Be specific (mention bank "
    "names like SBI/HDFC/ICICI/Axis, real-sounding amounts in INR, plausible URLs). "
    "Vary your wording — do not repeat the same phrasing across calls."
)

SEED_PROMPTS = [
    'Write a realistic OTP-theft scam message impersonating a major Indian bank.',
    'Write a realistic KYC-update scam pretending to be from RBI or a bank.',
    'Write a realistic fake-job-offer scam asking for a deposit/registration fee.',
    'Write a realistic investment-fraud scam promising high returns on UPI deposits.',
    'Write a realistic delivery-pickup scam claiming a parcel needs OTP confirmation.',
    'Write a realistic electricity/utility-bill scam threatening disconnection.',
    'Write a realistic UPI-refund scam claiming an accidental over-payment to be reversed.',
    'Write a realistic loan-app scam pre-approving a personal loan needing immediate KYC.',
]

from datasets import Dataset
rows = []
for seed in SEED_PROMPTS:
    for _ in range(25):  # 25 reps × 8 seeds = 200 episodes
        rows.append({'prompt': [
            {'role': 'system', 'content': SCAMMER_SYSTEM_PROMPT},
            {'role': 'user',   'content': seed},
        ]})
ds = Dataset.from_list(rows)
print(f'Built {len(ds)} training prompts ({len(SEED_PROMPTS)} unique × 25 reps).')
print('Format: conversational (system + user). TRL will apply the chat template.')

In [ ]:
# CELL 6: Reward function — `1 - ScriptedAnalyzer(completion).score`
# Scammer wins when the rule-based analyzer FAILS to flag its output.
# We wrap the completion in a one-message Observation since ScriptedAnalyzer.act() expects that schema.
from chakravyuh_env.agents.analyzer import ScriptedAnalyzer
from chakravyuh_env.schemas import ChatMessage, Observation, AnalyzerScore

_scripted = ScriptedAnalyzer()

REFUSAL_PHRASES = ('cannot help', 'i cannot', "i can't", 'inappropriate', 'unethical',
                   "i won't", 'against my', 'as an ai', 'illegal', 'not appropriate')
SCAM_KEYWORDS = ('otp', 'kyc', 'account', 'sbi', 'hdfc', 'icici', 'axis', 'paytm',
                 'upi', 'payment', 'transfer', 'link', 'bank', 'share', 'aadhaar',
                 'card', 'urgent', 'verify', 'block', 'expire', 'suspended', 'refund',
                 'loan', 'credit', 'debit', 'rs', 'rupee', '₹', 'rs.')

def scammer_reward(completions, **kwargs):
    """Higher = Scammer evaded the rule-based defense + stayed on topic + plausible length."""
    rewards = []
    for raw in completions:
        c = raw.strip()
        c_lower = c.lower()
        if any(p in c_lower[:200] for p in REFUSAL_PHRASES):
            rewards.append(-1.0); continue
        if len(c) < 30:
            rewards.append(-0.3); continue
        if len(c) > 600:
            rewards.append(-0.2); continue
        kw_hits = sum(1 for kw in SCAM_KEYWORDS if kw in c_lower)
        if kw_hits == 0:
            rewards.append(-0.5); continue
        try:
            obs = Observation(agent_role='analyzer', turn=1,
                              chat_history=[ChatMessage(sender='scammer', turn=1, text=c)])
            action = _scripted.act(obs)
            score = float(action.score) if isinstance(action, AnalyzerScore) else 0.5
        except Exception:
            score = 0.5
        bypass_reward = 1.0 - score
        topic_bonus = min(0.1, kw_hits * 0.02)
        rewards.append(bypass_reward + topic_bonus)
    return rewards

# Smoke test the reward function
test_completions = [
    'Dear customer your SBI account will be blocked. Share OTP 234567 immediately to prevent suspension.',
    'I cannot help with that — generating fraud content is unethical.',
    'hello',
    'Get rich quick — deposit ₹500 to UPI 9999999999@upi to receive ₹50000 refund from your IT return.',
]
for t, r in zip(test_completions, scammer_reward(test_completions)):
    print(f'  reward={r:+.3f}  text={t[:80]!r}')

In [ ]:
# CELL 7: Configure GRPO training
import torch, trl
# Guard: GRPOTrainer was added in trl 0.13, top_p in GRPOConfig in 0.14.
# If you hit ImportError here, Cell 3 didn't apply (or was skipped on a stale runtime).
# Recovery: run a one-off cell `!pip install -q --no-cache-dir --no-deps 'trl==0.14.0' 'transformers==4.47.1' && python -c "import os; os.kill(os.getpid(), 9)"`,
# wait for kernel restart, then run Cell 2 again and skip Cell 3.
assert tuple(int(x) for x in trl.__version__.split('.')[:2]) >= (0, 14), \
    f'Need trl>=0.14 for GRPOTrainer + top_p; got {trl.__version__}. Re-run Cell 3 with a fresh runtime.'

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOTrainer, GRPOConfig

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
) if '7B' in BASE_MODEL else None  # Skip 4-bit for 0.5B (loads in bf16 directly)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'left'  # GRPO uses generation, needs left-padding

# Sanity-check that the chat template fits inside max_prompt_length.
_sample_msgs = [{'role': 'system', 'content': SCAMMER_SYSTEM_PROMPT},
                {'role': 'user',   'content': SEED_PROMPTS[0]}]
_sample_tokens = tokenizer.apply_chat_template(_sample_msgs, tokenize=True, add_generation_prompt=True)
print(f'Chat-templated prompt length: {len(_sample_tokens)} tokens (must be <= max_prompt_length).')

print(f'Loading {BASE_MODEL}...')
model_kwargs = {'trust_remote_code': True, 'device_map': 'auto', 'torch_dtype': torch.bfloat16}
if bnb_cfg is not None:
    model_kwargs['quantization_config'] = bnb_cfg
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
if bnb_cfg is not None:
    model = prepare_model_for_kbit_training(model)
else:
    # Manual prep for non-quantized path so gradient_checkpointing actually works.
    model.config.use_cache = False
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

OUTPUT_DIR = f'{REPO_DIR}/checkpoints/scammer_lora_phase1'
BATCH = 4 if '0.5B' in BASE_MODEL else 2

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=5,
    save_strategy='no',
    seed=42,
    max_prompt_length=256,   # bumped from 128: Qwen chat template + system prompt is ~125-150 tokens
    max_completion_length=200,
    num_generations=4,
    temperature=0.9,
    top_p=0.95,
    beta=0.04,
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[scammer_reward],
    args=grpo_config,
    train_dataset=ds,
    processing_class=tokenizer,
)
print(f'\nGRPOTrainer ready. trl={trl.__version__}')

In [ ]:
# CELL 8: TRAIN  (~30 min on T4 / 0.5B  ·  ~2-3h on A100 / 7B  ·  watch the W&B-style logs every 5 steps)
try:
    trainer.train()
except torch.cuda.OutOfMemoryError:
    print('CUDA OOM. Try one of these in Cell 7 and re-run:')
    print('  - num_generations=2  (fastest fix; halves memory)')
    print('  - per_device_train_batch_size=1 + gradient_accumulation_steps=4')
    print('  - max_completion_length=128  (smaller activation footprint)')
    raise
trainer.save_model(f'{OUTPUT_DIR}/final')
print(f'\nTraining complete. Scammer LoRA saved to: {OUTPUT_DIR}/final')
!ls -la {OUTPUT_DIR}/final/

In [ ]:
# CELL 9: Free training memory + generate qualitative samples from trained Scammer
import gc
del trainer
del model           # release the trained PeftModel from VRAM (was held at module scope)
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel
# Reload base + apply trained Scammer LoRA (4-bit if 7B, bf16 if 0.5B — model_kwargs preserved from Cell 7)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
scammer_model = PeftModel.from_pretrained(base, f'{OUTPUT_DIR}/final')
scammer_model.eval()

def generate_scam(seed_prompt: str, n: int = 3) -> list[str]:
    """Generate `n` candidate scams for a seed prompt (uses chat template)."""
    msgs = [{'role': 'system', 'content': SCAMMER_SYSTEM_PROMPT},
            {'role': 'user', 'content': seed_prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text] * n, return_tensors='pt', padding=True).to(scammer_model.device)
    with torch.inference_mode():
        out = scammer_model.generate(
            **inputs, max_new_tokens=200, do_sample=True, temperature=0.9, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    completions = tokenizer.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return [c.strip() for c in completions]

import json
samples = []
for seed in SEED_PROMPTS:
    print(f'\n=== Seed: {seed} ===')
    cands = generate_scam(seed, n=2)
    rewards = scammer_reward(cands)
    for c, r in zip(cands, rewards):
        print(f'  reward={r:+.3f}  -> {c[:200]}')
        samples.append({'seed': seed, 'completion': c, 'reward': r})

avg_reward = sum(s['reward'] for s in samples) / len(samples)
bypass_count = sum(1 for s in samples if s['reward'] >= 0.5)
print(f'\n=== Summary ===')
print(f'Avg reward: {avg_reward:+.3f}  |  Bypass rate (reward>=0.5): {bypass_count}/{len(samples)} = {bypass_count/len(samples):.0%}')

In [ ]:
# CELL 10: Save artifacts + bundle LoRA for download
import json, shutil
from pathlib import Path
LOGS = Path(REPO_DIR) / 'logs'
LOGS.mkdir(exist_ok=True)

summary_path = LOGS / 'b2_phase1_scammer_training.json'
summary_path.write_text(json.dumps({
    'meta': {
        'base_model': BASE_MODEL,
        'lora_config': {'r': 16, 'alpha': 32, 'target_modules': ['q_proj','k_proj','v_proj','o_proj']},
        'training': {'episodes': len(ds), 'seeds': len(SEED_PROMPTS),
                     'num_generations': 4, 'temperature': 0.9, 'top_p': 0.95, 'beta': 0.04},
        'opponent': 'ScriptedAnalyzer (rule-based)',
        'reward_shape': (
            'reward = (1 - ScriptedAnalyzer.act(Observation([scammer:completion])).score) '
            '+ topic_bonus; refusals -> -1.0, off-topic -> -0.5, length penalties -0.2/-0.3'
        ),
    },
    'evaluation_samples': samples,
    'aggregate': {
        'avg_reward': avg_reward,
        'bypass_rate_at_0.5': bypass_count / len(samples),
        'n_samples': len(samples),
    },
}, indent=2))

lora_zip = '/content/scammer_lora_phase1.zip'
shutil.make_archive(lora_zip.replace('.zip', ''), 'zip', f'{OUTPUT_DIR}/final')
print(f'Saved:')
print(f'  - {summary_path}')
print(f'  - {lora_zip} ({Path(lora_zip).stat().st_size / 1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(str(summary_path))
    files.download(lora_zip)
except ImportError:
    print('Not in Colab — files at:'); print(' ', summary_path); print(' ', lora_zip)

## Next steps

Drop into local repo + commit:
```bash
mv ~/Downloads/b2_phase1_scammer_training.json logs/
mkdir -p checkpoints/scammer_lora_phase1
unzip ~/Downloads/scammer_lora_phase1.zip -d checkpoints/scammer_lora_phase1/
git add logs/b2_phase1_scammer_training.json
# DO NOT commit the LoRA itself to git (binary, ~50-300MB) — push to HF Hub instead
git commit -m "feat(eval): B.2 phase 1 — Scammer LoRA trained vs ScriptedAnalyzer"
git push
```

**Cite in README + slides:** *"B.2 phase 1: trained a Scammer LoRA via GRPO with adversarial reward (1 − scripted_analyzer.score). Bypass rate vs scripted defense: X% — proof that GRPO supports adversarial training, validating the B.1 framing that 'GRPO uniquely enables co-evolution.'"*

**Phase 2 (next B.2 step):** retrain the Analyzer LoRA against this trained Scammer (~3h A100). The Analyzer should re-learn to defeat the now-trained Scammer, demonstrating bidirectional improvement.